# ARC-AGI-2 Misfit Agent — Tier-1 Priors-Only Submission

**Disclosure (Tier-1):** This agent uses only Spelke Core Knowledge priors
(objectness, geometry, topology, numerosity, spatial). No LLMs, no
pretrained weights, no public-corpus heuristics, no internet access during
inference. Rule templates (`Identity`, `Translate2`, `Recolor`) are induced
from THIS task's train pairs only.

- Substrate: shared with the ARC-AGI-3 Misfit Agent
  (`perceptor`, `fingerprint`, `resonance`).
- Rule layer: re-targeted for static (input, output) pair induction.
- Honest abstain: if no rule fits all train pairs, attempts fall back to
  the identity baseline (test_input unchanged).
- Wall-clock self-kill: 8h30m total, 30s per task soft cap.
- Two attempts per test input: best program + second-best distinct program.

Repository: `AtomEons/arc-agi-3-misfit-agent`
Author: Atom McCree (AtomEons Systems Laboratory)


In [ ]:
# Cell 1 — offline pip install from /kaggle/input wheels
# Wheels must be bundled into the Kaggle dataset before submission.
import subprocess, sys, os, pathlib

WHEEL_DIR = '/kaggle/input/misfit-agent-wheels'
if os.path.isdir(WHEEL_DIR):
    wheels = sorted(pathlib.Path(WHEEL_DIR).glob('*.whl'))
    if wheels:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '--no-index',
            f'--find-links={WHEEL_DIR}',
            *[str(w) for w in wheels],
        ])
    else:
        print('No wheels found in', WHEEL_DIR, '- relying on Kaggle defaults')
else:
    print('Wheel dir not present - relying on Kaggle defaults (numpy bundled)')


In [ ]:
# Cell 2 — copy the substrate package into the working dir and slim it for
# the ARC-AGI-2 wrapper (we only need the arc2_* modules + perceptor).
import shutil, sys, pathlib

SRC = pathlib.Path('/kaggle/input/misfit-agent-source/src/misfit_agent')
DST = pathlib.Path('/kaggle/working/misfit_agent')
if SRC.exists():
    if DST.exists():
        shutil.rmtree(DST)
    shutil.copytree(SRC, DST)
    print('Copied substrate from', SRC)
else:
    raise RuntimeError(f'Substrate source not found at {SRC}. '
                       'Upload the misfit-agent-source Kaggle dataset first.')

# Slim __init__.py so importing misfit_agent does not pull in arcengine
# (arcengine is ARC-AGI-3 only; ARC-AGI-2 doesn't need it).
slim_init = '''"""misfit_agent — slim ARC-AGI-2 entry."""\n__all__ = []\n'''
(DST / '__init__.py').write_text(slim_init, encoding='utf-8')

sys.path.insert(0, '/kaggle/working')
print('sys.path[0] =', sys.path[0])


In [ ]:
# Cell 3 — run arc2_runner and write submission.json
import json, pathlib
from misfit_agent.arc2_runner import load_challenges, run

# Kaggle ARC-AGI-2 challenges live at one of these standard paths.
CANDIDATES = [
    '/kaggle/input/arc-prize-2026/arc-agi_evaluation_challenges.json',
    '/kaggle/input/arc-prize-2026/arc-agi_test_challenges.json',
    '/kaggle/input/arc-prize-2025/arc-agi_evaluation_challenges.json',
    '/kaggle/input/arc-prize-2024/arc-agi_evaluation_challenges.json',
]
challenges_path = next((p for p in CANDIDATES if pathlib.Path(p).exists()), None)
if challenges_path is None:
    raise RuntimeError('No ARC-AGI-2 challenges file found in standard Kaggle paths')

print('Loading', challenges_path)
challenges = load_challenges(challenges_path)
print('Loaded', len(challenges), 'tasks')

out = pathlib.Path('/kaggle/working/submission.json')
submission = run(
    challenges,
    out_path=out,
    wall_clock_seconds=8 * 60 * 60 + 30 * 60,
    per_task_seconds=30.0,
    verbose=False,
    write=True,
)
print('Wrote', out, 'with', len(submission), 'task entries')

# Quick shape sanity check
first = next(iter(submission.values()))
assert isinstance(first, list) and 'attempt_1' in first[0] and 'attempt_2' in first[0]
print('Submission shape sanity: OK')
